# Projeto **Motor Vehicle Collisions (Crashes) de Nova York** 

- Dataset atualizado diariamente com dados geográficos, temporais e categóricos ricos (causas do acidente, tipos de veículos, feridos, mortos).
- **Etapa 1:** Extração, Ingestão, Tratamento e Exibição com solução/arquitetura local.
- **Etapa 2:** Extração, Ingestão, Tratamento e Exibição com solução/arquitetura em nuvem (GCP).

---
# ETAPA 1
## 1.1 Stack Tecnológica (Ferramentas)

- **EXTRAÇÃO, TRANSFORMAÇÃO E CARGA (ETL):**
       - **VSCODE + PYTHON + AGENDADOR DE TAREFAS**

- **ARMAZENAMENTO:** 
       - **POSTGRES/POSTGIS**

- **VISUALIZAÇÃO (BI):** 
       - **FRAMEWORK DASH**

## 1.2 Arquitetura de Dados (Medallion Architecture Simplificada)

- O Processo de preparação dos dados foi organizado usando 3 camadas: 
       - **BRONZE:** Envolve o dado bruto baixado e/ou consumido da API
       - **SILVER:** Envolve o dado limpo e tipado dentro do banco de dados
       - **GOLD:** Envolve views de dados agregados, prontos para elaboração de visualizações pelo BI (GOLD)

```text
[NYC Open Data CSV] ──> Download Inicial
[NYC Open Data API] ──> Rotina de Atualização
       │
       │(Python Schedule)
       │
       ▼ 
[NYCdata\data\bronze_raw] ──> Camada Bronze (Arquivos JSON/CSV brutos)
       │
       │(Python Schedule)
       │
       ▼ 
[POSTGRES/POSTGIS] ──> Camada Silver (Dados limpos e tipados)
       │
       │(SQL/Views)
       │
       ▼ 
[POSTGRES/POSTGIS] ──> Camada Gold (Views de Dados agregados para o BI)
       │
       │(Python Script)
       │
       ▼
[FRAMEWORK PLOTLY DASH]

```

### Passo a Passo do Fluxo de Dados:

1. **Camada Bronze (Raw/Bruto):** 
- Download de arquivo .csv contendo dados de 01/01/2020 até 21/05/2026 do dataset **Motor Vehicle Collisions - Crashes**
- Script Python faz a requisição na API do NYC Open Data (Socrata API), baixa os novos dados (em formato CSV) e salva na tabela XXX do Banco de Dados **nyc_crashes_gis_db** Postgres/PostGIS em Docker.
2. **Camada Silver (Cleaned/Limpo):**
- O BigQuery lê o arquivo do Cloud Storage. Aqui você roda um script SQL para limpar o dado:
- Converter strings de data/hora em formatos `TIMESTAMP` reais.
- Tratar valores nulos (ex: causas de acidentes não especificadas).
- Filtrar coordenadas geográficas inválidas.
3. **Camada Gold (Analytics/Agregado):**
- Em vez de conectar o Looker Studio direto na tabela Silver de milhões de linhas (o que deixaria o painel lento e o custo de consulta do BigQuery alto), você criará **Views** ou tabelas agregadas no BigQuery.
- *Exemplo de View Gold:* Uma tabela que já calcula o total de acidentes e feridos consolidado por Bairro (*Borough*) e por Mês/Ano.

# ETAPA 2
## 1. Stack Tecnológica (Ferramentas)

- **Extração, transformação e Carga (ETL):
       - **Python** rodando localmente (fase de desenvolvimento) e **Cloud Functions** (fase de produção) + **Cloud Scheduler** para agendar a carga diária.

- **Armazenamento e Processamento:** **Google Cloud Storage (GCS)** (Data Lake bruto) + **Google BigQuery** (Data Warehouse e Camada de Analytics).

- **Visualização (BI):** **Google Looker Studio**.

---

## 2. Arquitetura de Dados (Medallion Architecture Simplificada)

Para demonstrar maturidade como desenvolvedor, você não vai apenas jogar o dado no BigQuery. Vamos organizar o fluxo em 3 camadas lógicas dentro do GCP:

```text
[NYC Open Data API]
       │
       ▼ (Cloud Function)
[Google Cloud Storage] ──> Camada Bronze (Arquivos JSON/CSV brutos)
       │
       ▼ (BigQuery SQL)
[BigQuery - Dataset Silver] ──> Camada Silver (Dados limpos e tipados)
       │
       ▼ (BigQuery SQL / Views)
[BigQuery - Dataset Gold]   ──> Camada Gold (Dados agregados para o BI)
       │
       ▼
[Looker Studio Dashboard]

```

### Passo a Passo do Fluxo de Dados:

1. **Camada Bronze (Raw/Bruto):** Um script Python faz a requisição na API do NYC Open Data (Socrata API), baixa os novos dados (em formato JSON ou CSV) e salva diretamente em um bucket no **Google Cloud Storage**.
2. **Camada Silver (Cleaned/Limpo):** O BigQuery lê o arquivo do Cloud Storage. Aqui você roda um script SQL para limpar o dado:
* Converter strings de data/hora em formatos `TIMESTAMP` reais.
* Tratar valores nulos (ex: causas de acidentes não especificadas).
* Filtrar coordenadas geográficas inválidas.
3. **Camada Gold (Analytics/Agregado):** Em vez de conectar o Looker Studio direto na tabela Silver de milhões de linhas (o que deixaria o painel lento e o custo de consulta do BigQuery alto), você criará **Views** ou tabelas agregadas no BigQuery.
* *Exemplo de View Gold:* Uma tabela que já calcula o total de acidentes e feridos consolidado por Bairro (*Borough*) e por Mês/Ano.



---

## 3. Estratégia de Otimização e Custos no BigQuery (Crucial para Freelancers)

Clientes morrem de medo de sustos na fatura do BigQuery. Mostrar que você sabe controlar custos é um enorme diferencial de venda. No seu projeto, você aplicará:

* **Particionamento:** Particione a tabela Silver pela coluna de data do acidente (`CRASH_DATE`). Quando o Looker Studio filtrar por "Últimos 30 dias", o BigQuery só vai escanear os dados daqueles 30 dias, reduzindo o custo e aumentando a velocidade em 90%.
* **BI Engine no Looker Studio:** Ative o BI Engine (que possui um limite gratuito na GCP) para acelerar as consultas do Looker Studio guardando os dados em cache de memória.

---

## 4. O que o Dashboard no Looker Studio deve responder? (O Produto Final)

Para o seu portfólio de freelance, o painel precisa parecer uma solução de negócios. Divida o Looker Studio em 3 páginas:

1. **Visão Executiva (KPIs Gerais):**
* Total de acidentes no período selecionado.
* Índice de severidade (% de acidentes com vítimas ou mortes).
* Principais fatores contribuintes (distração do motorista, álcool, velocidade).


2. **Análise Temporal e de Frota:**
* Gráfico de linha mostrando a tendência de acidentes por mês/ano.
* Mapa de calor mostrando os horários e dias da semana mais perigosos (ex: sextas-feiras à noite).
* Tipos de veículos mais envolvidos (Sedans, SUVs, Caminhões de entrega).


3. **Visão Geográfica (Foco em Localização):**
* Mapa interativo com os pontos de concentração de acidentes (*Hotspots*).
* Filtro por bairro (*Borough*) ou código postal (*Zip Code*).


### LOAD DATA

In [7]:
import pandas as pd


def importar_csv_para_dataframe(caminho_arquivo, linhas_para_teste=None):
    """Carrega um arquivo CSV e o transforma em um DataFrame do Pandas."""
    try:
        print(f"⌛ Iniciando a leitura do arquivo: {caminho_arquivo}")

        # Se for teste, usa o motor padrão para aceitar nrows.
        # Se for carga total, ativa o pyarrow para máxima performance.
        if linhas_para_teste:
            df = pd.read_csv(caminho_arquivo, nrows=linhas_para_teste)
        else:
            df = pd.read_csv(caminho_arquivo, engine="pyarrow")

        print("✅ Arquivo importado com sucesso!")
        print(
            f"📊 Dimensões do DataFrame: {df.shape[0]} linhas e {df.shape[1]} colunas.\n"
        )

        return df

    except FileNotFoundError:
        print(
            f"❌ Erro: O arquivo no caminho '{caminho_arquivo}' não foi encontrado."
        )
        return None
    except Exception as e:
        print(f"❌ Ocorreu um erro inesperado: {e}")
        return None


# --- Execução de Teste ---
if __name__ == "__main__":
    # O 'r' antes das aspas transforma em Raw String, corrigindo as barras do Windows
    CAMINHO_ARQUIVO = r"C:\Users\HP\Documents\Projetos\GeoDev\NYCdata\data\bronze_raw\Motor_Vehicle_Collisions_20200101_20260521.csv"

    # Testando primeiro com apenas 1000 linhas para verificar a estrutura
    df_crashes = importar_csv_para_dataframe(
        CAMINHO_ARQUIVO, linhas_para_teste=1000
    )

    if df_crashes is not None:
        print("👀 Primeiras linhas do DataFrame:")
        display(df_crashes.head())

        print("\n🔍 Tipos de dados detectados nas colunas:")
        print(df_crashes.dtypes)

⌛ Iniciando a leitura do arquivo: C:\Users\HP\Documents\Projetos\GeoDev\NYCdata\data\bronze_raw\Motor_Vehicle_Collisions_20200101_20260521.csv
✅ Arquivo importado com sucesso!
📊 Dimensões do DataFrame: 1000 linhas e 29 colunas.

👀 Primeiras linhas do DataFrame:


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,01/02/2020,0:00,NaN,NaN,NaN,NaN,NaN,CROSS ISLAND PARKWAY,NaN,NaN,...,NaN,NaN,NaN,NaN,4267700,Sedan,NaN,NaN,NaN,NaN
1,01/02/2020,12:57,NaN,NaN,NaN,NaN,NaN,W 57 & 8th Ave,W 57,NaN,...,Unspecified,NaN,NaN,NaN,4268255,Taxi,Pick-up Truck,NaN,NaN,NaN
2,01/02/2020,20:24,NaN,NaN,NaN,NaN,NaN,HUTCHINSON RIVER PARKWAY,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4268404,Station Wagon/Sport Utility Vehicle,Sedan,NaN,NaN,NaN
3,01/02/2020,18:23,NaN,NaN,NaN,NaN,NaN,VAN WYCK EXPWY,NaN,NaN,...,Unspecified,Unspecified,Unspecified,NaN,4268152,Sedan,Sedan,Sedan,Sedan,NaN
4,01/02/2020,19:00,NaN,NaN,"40,699955","-73,98682","(40.699955, -73.98682)",JAY STREET,NaN,NaN,...,Passing or Lane Usage Improper,NaN,NaN,NaN,4268117,Sedan,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN



🔍 Tipos de dados detectados nas colunas:
CRASH DATE                           str
CRASH TIME                           str
BOROUGH                              str
ZIP CODE                         float64
LATITUDE                             str
LONGITUDE                            str
LOCATION                             str
ON STREET NAME                       str
CROSS STREET NAME                    str
OFF STREET NAME                      str
NUMBER OF PERSONS INJURED          int64
NUMBER OF PERSONS KILLED           int64
NUMBER OF PEDESTRIANS INJURED      int64
NUMBER OF PEDESTRIANS KILLED       int64
NUMBER OF CYCLIST INJURED          int64
NUMBER OF CYCLIST KILLED           int64
NUMBER OF MOTORIST INJURED         int64
NUMBER OF MOTORIST KILLED          int64
CONTRIBUTING FACTOR VEHICLE 1        str
CONTRIBUTING FACTOR VEHICLE 2        str
CONTRIBUTING FACTOR VEHICLE 3        str
CONTRIBUTING FACTOR VEHICLE 4        str
CONTRIBUTING FACTOR VEHICLE 5        str
COLLISION_ID   